# Notebook 04 — Recommendation System & Purchasing Trends Analysis
**Author:** Kevin  
**Role:** Developer — Recommendations & Trends  
**Task:** Load cleaned data from HDFS → Build item-based recommendations → Analyse purchasing trends → Save outputs

---
> **Prerequisite:** `01_data_cleaning.ipynb` must have been run first.

**Two parts:**
1. **Recommendation Logic** — item-based collaborative filtering using co-purchase counts
2. **Purchasing Trends** — monthly, quarterly, seasonal, and country-level trends

## 1. Load Packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.makedirs("output/recommendations", exist_ok=True)
os.makedirs("output/trends", exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Packages loaded.")

## 2. Start Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("RetailRecommendations") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 3. Load Cleaned Dataset from Hadoop HDFS

In [ ]:
# Load cleaned Parquet — saved by Notebook 01
df = spark.read.parquet("hdfs:///retail/cleaned/")

print(f"Loaded {df.count():,} records from HDFS.")
df.show(3, truncate=True)

# Register for Spark SQL use
df.createOrReplaceTempView("retail")

---
# PART 1 — Recommendation System
---

## 4. Build Customer-Product Purchase Matrix

For each customer, we record every product they have bought and how many times.  
This is the foundation for item-based recommendations.

In [ ]:
# Customer × Product purchase counts
customer_product = spark.sql("""
    SELECT
        CustomerID,
        StockCode,
        Description,
        SUM(Quantity)   AS TotalQtyBought,
        COUNT(Invoice)  AS TimesPurchased
    FROM retail
    GROUP BY CustomerID, StockCode, Description
""")

customer_product.createOrReplaceTempView("customer_product")
print(f"Customer-product matrix rows: {customer_product.count():,}")
customer_product.show(5, truncate=30)

## 5. Item-Based Recommendation Logic

**How it works:**  
1. Find all customers who bought Product A  
2. Look at everything else those customers also bought  
3. Score each candidate product by how many of those customers bought it  
4. The higher the score, the stronger the recommendation  

**Design Pattern:** JOIN + GROUP BY (Aggregation) + ORDER BY (Sorting)

In [ ]:
def get_recommendations(product_code, top_n=10):
    """
    Given a StockCode, return the top N recommended products.
    Logic: Find customers who bought this product,
           then rank other products those customers also bought.
    """
    query = f"""
        SELECT
            b.StockCode                     AS RecommendedCode,
            b.Description                   AS RecommendedProduct,
            COUNT(DISTINCT b.CustomerID)    AS CustomerOverlap,
            SUM(b.TotalQtyBought)           AS TotalQtySold
        FROM customer_product a
        JOIN customer_product b
            ON  a.CustomerID = b.CustomerID
            AND b.StockCode <> '{product_code}'
        WHERE a.StockCode = '{product_code}'
        GROUP BY b.StockCode, b.Description
        ORDER BY CustomerOverlap DESC, TotalQtySold DESC
        LIMIT {top_n}
    """
    return spark.sql(query)

# Test with the most popular product in the dataset
top_product = spark.sql("""
    SELECT StockCode, Description, SUM(Quantity) AS TotalQty
    FROM retail
    GROUP BY StockCode, Description
    ORDER BY TotalQty DESC
    LIMIT 1
""").collect()[0]

test_code = top_product["StockCode"]
test_name = top_product["Description"]

print(f"Getting recommendations for: [{test_code}] {test_name}")
print()
recs = get_recommendations(test_code, top_n=10)
recs.show(10, truncate=40)

## 6. Generate Recommendations for Top 5 Products

In [ ]:
# Get top 5 products by total quantity sold
top5_products = spark.sql("""
    SELECT StockCode, Description, SUM(Quantity) AS TotalQty
    FROM retail
    GROUP BY StockCode, Description
    ORDER BY TotalQty DESC
    LIMIT 5
""").collect()

# Collect all recommendations into a single DataFrame
all_recs = []

for product in top5_products:
    code = product["StockCode"]
    name = product["Description"]
    print(f"\n--- Recommendations for: {name} ({code}) ---")

    rec_df = get_recommendations(code, top_n=5).toPandas()
    rec_df.insert(0, "InputProductCode", code)
    rec_df.insert(1, "InputProductName", name)
    all_recs.append(rec_df)

    for _, row in rec_df.iterrows():
        print(f"  → {row['RecommendedProduct'][:45]:<45} (overlap: {row['CustomerOverlap']})")  

# Combine into one DataFrame
recommendations_df = pd.concat(all_recs, ignore_index=True)
print(f"\nTotal recommendation rows: {len(recommendations_df)}")

## 7. Visualise — Recommendation Customer Overlap

In [ ]:
# Chart showing recommendation strength (customer overlap) for top product
top1_recs = recommendations_df[
    recommendations_df["InputProductCode"] == top5_products[0]["StockCode"]
].head(10)

fig, ax = plt.subplots(figsize=(11, 5))
colors = sns.color_palette("Blues_r", len(top1_recs))

short_names = top1_recs["RecommendedProduct"].str[:30]
ax.barh(short_names[::-1], top1_recs["CustomerOverlap"][::-1].values, color=colors)

ax.set_xlabel("Number of Customers Who Also Bought This", fontsize=12)
ax.set_title(
    f"Top 10 Recommendations for:\n'{test_name[:50]}'",
    fontsize=13, fontweight="bold"
)

plt.tight_layout()
plt.savefig("output/recommendations/chart_recommendations.png", bbox_inches="tight")
plt.show()
print("Chart saved.")

---
# PART 2 — Purchasing Trends Analysis
---

## 8. Monthly Sales Trends — Revenue & Volume Over Time

In [ ]:
# Monthly aggregation using Spark SQL
monthly = spark.sql("""
    SELECT
        YearMonth,
        Year,
        Month,
        ROUND(SUM(TotalPrice), 2)      AS Revenue,
        COUNT(DISTINCT Invoice)         AS OrderCount,
        COUNT(DISTINCT CustomerID)      AS UniqueCustomers,
        ROUND(AVG(TotalPrice), 2)       AS AvgOrderValue
    FROM retail
    GROUP BY YearMonth, Year, Month
    ORDER BY YearMonth
""").toPandas()

monthly.to_csv("output/trends/monthly_trends.csv", index=False)

# Plot — 2-row chart
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# Revenue
axes[0].plot(monthly["YearMonth"], monthly["Revenue"],
             color="#1F4E79", marker="o", linewidth=2)
axes[0].fill_between(monthly["YearMonth"], monthly["Revenue"], alpha=0.15, color="#2E75B6")
axes[0].set_ylabel("Revenue (£)", fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}K"))
axes[0].set_title("Monthly Revenue Trend", fontsize=13, fontweight="bold")

# Orders & Customers
axes[1].bar(monthly["YearMonth"], monthly["OrderCount"],
            color="#2E75B6", alpha=0.7, label="Orders")
ax2 = axes[1].twinx()
ax2.plot(monthly["YearMonth"], monthly["UniqueCustomers"],
         color="#E07B39", marker="s", linewidth=2, label="Unique Customers")
axes[1].set_ylabel("Order Count", fontsize=11)
ax2.set_ylabel("Unique Customers", fontsize=11, color="#E07B39")
axes[1].set_title("Monthly Orders & Customer Count", fontsize=13, fontweight="bold")

for ax in axes:
    ax.set_xticklabels(monthly["YearMonth"], rotation=45, ha="right", fontsize=8)
    ax.xaxis.set_tick_params(which="both", labelbottom=True)

plt.tight_layout()
plt.savefig("output/trends/chart_monthly_trends.png", bbox_inches="tight")
plt.show()
print("Monthly trends chart saved.")

**Interpretation:** Revenue and order count both peak in November each year, driven by Christmas retail demand. Both metrics are consistently higher in 2010 than 2009 across all months, indicating strong year-on-year business growth.

## 9. Quarterly Revenue Comparison — Seasonal Analysis

In [ ]:
# Assign quarter labels
quarterly = spark.sql("""
    SELECT
        Year,
        CASE
            WHEN Month IN (1,2,3)   THEN 'Q1 (Jan-Mar)'
            WHEN Month IN (4,5,6)   THEN 'Q2 (Apr-Jun)'
            WHEN Month IN (7,8,9)   THEN 'Q3 (Jul-Sep)'
            WHEN Month IN (10,11,12) THEN 'Q4 (Oct-Dec)'
        END AS Quarter,
        ROUND(SUM(TotalPrice), 2)     AS Revenue,
        COUNT(DISTINCT Invoice)        AS OrderCount
    FROM retail
    GROUP BY Year, Quarter
    ORDER BY Year, Quarter
""").toPandas()

quarterly.to_csv("output/trends/quarterly_trends.csv", index=False)

# Plot grouped bars
fig, ax = plt.subplots(figsize=(11, 5))

quarter_labels = quarterly["Quarter"].unique()
years = quarterly["Year"].unique()
x = np.arange(len(quarter_labels))
width = 0.35

colors = ["#1F4E79", "#2E75B6", "#7FB3D3"]
for i, year in enumerate(sorted(years)):
    year_data = quarterly[quarterly["Year"] == year]
    # Align to quarter order
    vals = [year_data[year_data["Quarter"] == q]["Revenue"].values[0]
            if q in year_data["Quarter"].values else 0
            for q in quarter_labels]
    ax.bar(x + i * width - width/2, vals, width, label=str(year),
           color=colors[i % len(colors)], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(quarter_labels, fontsize=11)
ax.set_ylabel("Revenue (£)", fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e6:.1f}M"))
ax.set_title("Quarterly Revenue Comparison by Year", fontsize=14, fontweight="bold")
ax.legend(title="Year", fontsize=11)

plt.tight_layout()
plt.savefig("output/trends/chart_quarterly.png", bbox_inches="tight")
plt.show()
print("Quarterly chart saved.")

**Interpretation:** Q4 (Oct–Dec) is by far the highest revenue quarter in both years, confirming strong Christmas seasonality. Q1 (Jan–Mar) is consistently the lowest, showing a post-holiday slowdown. In 2010, all quarters show higher revenue than 2009, reflecting business growth.

## 10. Top Products per Country — Purchasing Preferences

In [ ]:
# Target countries for comparison
target_countries = ["United Kingdom", "Germany", "France"]

# Rank products within each country using a window function
country_product_rank = spark.sql("""
    SELECT
        Country,
        Description,
        SUM(Quantity) AS TotalQty
    FROM retail
    WHERE Country IN ('United Kingdom', 'Germany', 'France')
    GROUP BY Country, Description
""")

# Use Window to rank per country
window = Window.partitionBy("Country").orderBy(F.desc("TotalQty"))
country_top5 = country_product_rank \
    .withColumn("Rank", F.rank().over(window)) \
    .filter(F.col("Rank") <= 5) \
    .orderBy("Country", "Rank") \
    .toPandas()

country_top5.to_csv("output/trends/top5_products_by_country.csv", index=False)
print(country_top5[["Country", "Rank", "Description", "TotalQty"]].to_string(index=False))

In [ ]:
# Plot top 5 products for each country side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = sns.color_palette("Blues_r", 5)

for i, country in enumerate(target_countries):
    cdata = country_top5[country_top5["Country"] == country]
    short = cdata["Description"].str[:25]
    axes[i].barh(short[::-1].values, cdata["TotalQty"][::-1].values, color=palette)
    axes[i].set_title(country, fontsize=13, fontweight="bold")
    axes[i].set_xlabel("Quantity Sold", fontsize=10)
    if i == 0:
        axes[i].set_ylabel("Product", fontsize=10)

plt.suptitle("Top 5 Products by Country", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("output/trends/chart_top5_by_country.png", bbox_inches="tight")
plt.show()
print("Country products chart saved.")

**Interpretation:** Top products vary noticeably by country, suggesting that product preferences and buying volumes differ across markets. UK customers dominate in volume for almost all categories. Germany and France show interest in different product types, suggesting the retailer should tailor marketing messages per market.

## 11. Average Order Value Trend — Month Over Month

In [ ]:
# Average order value per month
aov_trend = spark.sql("""
    SELECT
        YearMonth,
        ROUND(SUM(TotalPrice) / COUNT(DISTINCT Invoice), 2) AS AvgOrderValue,
        ROUND(SUM(Quantity) / COUNT(DISTINCT Invoice), 1)   AS AvgItemsPerOrder
    FROM retail
    GROUP BY YearMonth
    ORDER BY YearMonth
""").toPandas()

aov_trend.to_csv("output/trends/avg_order_value_trend.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(aov_trend["YearMonth"], aov_trend["AvgOrderValue"],
        color="#1F4E79", marker="o", linewidth=2.5)
ax.fill_between(aov_trend["YearMonth"], aov_trend["AvgOrderValue"], alpha=0.12, color="#2E75B6")
ax.set_ylabel("Avg Order Value (£)", fontsize=12)
ax.set_title("Average Order Value — Month Over Month", fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("output/trends/chart_avg_order_value.png", bbox_inches="tight")
plt.show()
print("AOV trend chart saved.")

## 12. Customer Retention — Repeat vs One-Time Buyers

In [ ]:
# Classify customers as one-time or repeat buyers
customer_orders = spark.sql("""
    SELECT
        CustomerID,
        COUNT(DISTINCT Invoice)           AS OrderCount,
        ROUND(SUM(TotalPrice), 2)         AS TotalSpend,
        MIN(InvoiceDate)                  AS FirstOrder,
        MAX(InvoiceDate)                  AS LastOrder
    FROM retail
    GROUP BY CustomerID
""").toPandas()

customer_orders["CustomerType"] = customer_orders["OrderCount"].apply(
    lambda x: "One-Time Buyer" if x == 1 else
              "Repeat Buyer (2-5)" if x <= 5 else
              "Loyal Customer (6+)"
)

type_summary = customer_orders.groupby("CustomerType").agg(
    CustomerCount=("CustomerID", "count"),
    AvgSpend=("TotalSpend", "mean")
).reset_index()

customer_orders.to_csv("output/trends/customer_retention.csv", index=False)

# Pie chart + bar chart side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors_pie = ["#1F4E79", "#2E75B6", "#7FB3D3"]
axes[0].pie(type_summary["CustomerCount"], labels=type_summary["CustomerType"],
            autopct="%1.1f%%", startangle=140,
            colors=colors_pie, pctdistance=0.8)
axes[0].set_title("Customer Segments by Purchase Frequency", fontsize=12, fontweight="bold")

axes[1].bar(type_summary["CustomerType"], type_summary["AvgSpend"],
            color=colors_pie, edgecolor="white")
axes[1].set_ylabel("Average Total Spend (£)", fontsize=11)
axes[1].set_title("Avg Total Spend by Customer Type", fontsize=12, fontweight="bold")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[1].set_xticklabels(type_summary["CustomerType"], rotation=15, ha="right")

plt.tight_layout()
plt.savefig("output/trends/chart_customer_retention.png", bbox_inches="tight")
plt.show()
print(type_summary.to_string(index=False))
print("Retention chart saved.")

**Interpretation:** Loyal customers (6+ orders) represent a smaller portion of the customer base but generate significantly higher total spend per customer. Even though one-time buyers may be the majority by count, loyal customers contribute disproportionately to revenue — making retention strategies highly valuable.

## 13. Save All Results — HDFS & Local

In [ ]:
# ── Save recommendations to HDFS ─────────────────────────────────────────────
# Build Spark DF from pandas results and save
recs_spark = spark.createDataFrame(recommendations_df)
recs_spark.write.parquet(
    "hdfs:///retail/output/recommendations/",
    mode="overwrite"
)
print("Recommendations saved to HDFS: hdfs:///retail/output/recommendations/")

# ── Save trends to HDFS ───────────────────────────────────────────────────────
spark.createDataFrame(monthly).write.parquet(
    "hdfs:///retail/output/trends/monthly/",
    mode="overwrite"
)
spark.createDataFrame(quarterly).write.parquet(
    "hdfs:///retail/output/trends/quarterly/",
    mode="overwrite"
)
print("Trends saved to HDFS: hdfs:///retail/output/trends/")

# ── Save recommendations locally (CSV) ───────────────────────────────────────
recommendations_df.to_csv("output/recommendations/recommendations_top5_products.csv", index=False)
print("CSV saved locally: output/recommendations/recommendations_top5_products.csv")

print("\nAll files saved. Notebook 04 complete.")

In [ ]:
# List all local output files
print("Local output files:")
for root, dirs, files in os.walk("output"):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path:<60} ({size:,} bytes)")

---
## Summary

### Part 1 — Recommendations
| Step | Action | Design Pattern |
|------|--------|----------------|
| 1 | Build customer-product purchase matrix | Aggregation |
| 2 | JOIN matrix with itself on CustomerID | JOIN |
| 3 | Count customer overlap per product pair | Aggregation |
| 4 | ORDER BY overlap score DESC | Sorting |
| 5 | Return top N recommendations | Top-K / Filtering |

### Part 2 — Trends
| Step | Action | Key Finding |
|------|--------|-------------|
| Monthly trend | Revenue & orders by month | Nov peak, Q4 strongest |
| Quarterly | Q1–Q4 per year | Q4 >> Q1, 2010 > 2009 |
| Country products | Top 5 per country | UK dominates, markets differ |
| AOV trend | Avg order value over time | Seasonal spikes visible |
| Retention | One-time vs loyal buyers | Loyal = higher value |

**All outputs saved to:** `output/recommendations/` and `output/trends/`  
**HDFS paths:** `hdfs:///retail/output/recommendations/` and `hdfs:///retail/output/trends/`